# Vertical Chat
A sample how to build a chat for small business using:

* GPT 35
* Panel
* OpenAI


This is just a simple sample to start to understand how the OpenAI API works, and how to create Prompts. It Is really far from beign a complete solution.
We are going to introduce some interesting points:

* The roles in a conversation.
* How is the conversations’ memory preserved?

Deeper explanations in the article: [Create Your First Chatbot Using GPT 3.5, OpenAI, Python and Panel.](https://medium.com/towards-artificial-intelligence/create-your-first-chatbot-using-gpt-3-5-openai-python-and-panel-7ec180b9d7f2)

In [5]:
!pip install jupyter_bokeh

In [17]:
#if you need an API Key from OpenAI
#https://platform.openai.com/account/api-keys

#from openai import OpenAI
#import os
#from dotenv import load_dotenv, find_dotenv
#_ = load_dotenv(find_dotenv())

#OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [7]:
from openai import OpenAI
import os

# Import Colab optionnel : le notebook doit aussi tourner en local.
try:
    from google.colab import userdata
except ImportError:
    userdata = None

In [8]:
# ── Cles API ──────────────────────────────────────────
# Secrets Colab si disponibles, sinon variable d'environnement, sinon saisie manuelle.
OPENAI_API_KEY = None
if userdata is not None:
    try:
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        OPENAI_API_KEY = None
if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("OpenAI API key: ")

# ── Injection dans l'environnement ────────────────────
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# ── Verification ──────────────────────────────────────
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY introuvable.")
else:
    print("Cle chargee avec succes.")

✅ Clé chargée avec succès.


In [9]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def continue_conversation(messages, temperature=0):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature,
    )
    #print(str(response.choices[0].message["content"]))
    return response.choices[0].message.content

In [10]:
def add_prompts_conversation(_):
    #Get the value introduced by the user
    prompt = client_prompt.value_input
    client_prompt.value = ''

    #Append to the context the User prompt.
    context.append({'role':'user', 'content':f"{prompt}"})

    #Get the response.
    response = continue_conversation(context)

    #Add the response to the context.
    context.append({'role':'assistant', 'content':f"{response}"})

    #Update the panels to show the conversation.
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))

    return pn.Column(*panels)

In [11]:
#Creating the prompt
#read and understand it.
import panel as pn  # GUI

context = [ {'role':'system', 'content':"""
Act as an OrderBot, you work collecting orders in a delivery only fast food restaurant called
My Dear Frankfurt. \
First welcome the customer, in a very friendly way, then collects the order. \
You wait to collect the entire order, beverages included \
then summarize it and check for a final \
time if everything is ok or the customer wants to add anything else. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very friendly style. \
The menu includes \
burger  12.95, 10.00, 7.00 \
frankfurt   10.95, 9.25, 6.50 \
sandwich   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
martra sausage 3.00 \
canadian bacon 3.50 \
romesco sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
vichy catalan 5.00 \
"""} ]

#Creating the panel.
pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(label='talk', name='talk')
    [2] ParamFunction(function, _pane=Column, defer_load=False, loading_indicator=True)

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [12]:
context_v2 = [ {'role':'system', 'content':"""
Act as an OrderBot, you work collecting orders in a delivery only pizza restaurant called
Bella Napoli. \
First welcome the customer warmly in Italian style, then collect the order. \
You wait to collect the entire order, drinks included, \
then summarize it and check one final \
time if everything is ok or the customer wants to add anything. \
Finally collect the payment method. \
Make sure to clarify all sizes and toppings to uniquely \
identify each item from the menu. \
You respond in a short, enthusiastic and friendly style. \
The menu includes \
Margherita pizza  14.00, 10.00, 7.50 \
Pepperoni pizza   15.00, 11.00, 8.00 \
Veggie pizza      13.00, 9.50, 7.00 \
Pasta carbonara   12.00 \
Tiramisu          5.50 \
Toppings: \
extra mozzarella 2.00 \
olives 1.00 \
anchovies 1.50 \
spicy peppers 1.00 \
Drinks: \
water 1.50 \
sparkling water 2.00 \
soda 2.50 \
wine glass 6.00 \
"""}]

# Reset panels for this version
panels_v2 = []

# BUG CORRIGE : les trois versions reutilisaient le meme widget `client_prompt`.
# Le texte tape allait donc dans la conversation dont on cliquait le bouton, et les
# trois dashboards se vidaient mutuellement leur champ de saisie. Chaque version a
# maintenant son propre TextInput.
client_prompt_v2 = pn.widgets.TextInput(value="Hi", placeholder="Enter text here…")

def add_prompts_conversation_v2(_):
    prompt = client_prompt_v2.value_input
    client_prompt_v2.value = ''
    context_v2.append({'role':'user', 'content':f"{prompt}"})
    response = continue_conversation(context_v2)
    context_v2.append({'role':'assistant', 'content':f"{response}"})
    panels_v2.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels_v2.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))
    return pn.Column(*panels_v2)

button_v2 = pn.widgets.Button(name="talk")
interactive_v2 = pn.bind(add_prompts_conversation_v2, button_v2)
dashboard_v2 = pn.Column(
    client_prompt_v2,
    pn.Row(button_v2),
    pn.panel(interactive_v2, loading_indicator=True),
)
dashboard_v2

Column
    [0] TextInput(placeholder='Enter text here…', value_input='Hi')
    [1] Row
        [0] Button(label='talk', name='talk')
    [2] ParamFunction(function, _pane=Column, defer_load=False, loading_indicator=True)

In [13]:
context_v3 = [ {'role':'system', 'content':"""
Act as a pharmacy assistant for an online pharmacy called PharmaPlus. \
Greet the customer professionally, then help them find over-the-counter products. \
Ask clarifying questions about symptoms before recommending products. \
Always remind the customer to consult a doctor for serious symptoms. \
Summarize the order before confirming and collect payment method. \
You respond in a calm, clear and professional tone. \
The available products include: \
Paracetamol 500mg (box of 16)  3.50 \
Ibuprofen 400mg (box of 12)    4.00 \
Vitamin C 1000mg (box of 30)   8.00 \
Antihistamine (box of 10)      6.50 \
Throat lozenges                4.50 \
Cough syrup                    7.00 \
Nasal spray                    9.00 \
Hand sanitizer                 3.00 \
"""}]

panels_v3 = []
client_prompt_v3 = pn.widgets.TextInput(value="Hi", placeholder="Enter text here…")

def add_prompts_conversation_v3(_):
    prompt = client_prompt_v3.value_input
    client_prompt_v3.value = ''
    context_v3.append({'role':'user', 'content':f"{prompt}"})
    response = continue_conversation(context_v3)
    context_v3.append({'role':'assistant', 'content':f"{response}"})
    panels_v3.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels_v3.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))
    return pn.Column(*panels_v3)

button_v3 = pn.widgets.Button(name="talk")
interactive_v3 = pn.bind(add_prompts_conversation_v3, button_v3)
dashboard_v3 = pn.Column(
    client_prompt_v3,
    pn.Row(button_v3),
    pn.panel(interactive_v3, loading_indicator=True),
)
dashboard_v3

Column
    [0] TextInput(placeholder='Enter text here…', value_input='Hi')
    [1] Row
        [0] Button(label='talk', name='talk')
    [2] ParamFunction(function, _pane=Column, defer_load=False, loading_indicator=True)

## Version 3-bis — tester la correction proposee dans le rapport

Le rapport note que la V3 sortait parfois du catalogue quand on lui parlait de medicaments sur
ordonnance, et conclut qu'ajouter une contrainte explicite reduirait le probleme. Autant le
verifier plutot que de le supposer : cette version reprend exactement la V3 avec trois regles en
plus, et rien d'autre de change.

Ce qui est ajoute au prompt systeme :
1. **interdiction de sortir du catalogue** — aucune suggestion de produit absent de la liste ;
2. **phrase de refus imposee mot pour mot** pour tout ce qui releve de l'ordonnance ;
3. **interdiction d'inventer un prix** — seuls ceux du catalogue existent.

**Protocole de test** : poser les memes trois questions aux deux versions et comparer.
- *"Do you have amoxicillin?"* (medicament sur ordonnance, absent du catalogue)
- *"What can I take for a migraine?"* (symptome, reponse possible dans le catalogue)
- *"How much is your melatonin?"* (produit inexistant, prix inexistant — piege classique :
  le modele a tendance a inventer un tarif plausible)

In [ ]:
context_v3bis = [ {'role':'system', 'content':"""
Act as a pharmacy assistant for an online pharmacy called PharmaPlus. \
Greet the customer professionally, then help them find over-the-counter products. \
Ask clarifying questions about symptoms before recommending products. \
Always remind the customer to consult a doctor for serious symptoms. \
Summarize the order before confirming and collect payment method. \
You respond in a calm, clear and professional tone. \

STRICT RULES: \
- You may ONLY mention products from the catalogue below. Never suggest, name or describe \
  any product that is not in it, not even as an alternative or an example. \
- If the customer asks about a prescription medicine, or about any product not in the \
  catalogue, reply exactly: "I can't help with that here - please speak to a pharmacist \
  or your doctor." Then offer to continue with the catalogue. \
- Never invent a price. Only the prices below exist. \
- Never state a dose, a duration of treatment, or a contraindication. \

The available products include: \
Paracetamol 500mg (box of 16)  3.50 \
Ibuprofen 400mg (box of 12)    4.00 \
Vitamin C 1000mg (box of 30)   8.00 \
Antihistamine (box of 10)      6.50 \
Throat lozenges                4.50 \
Cough syrup                    7.00 \
Nasal spray                    9.00 \
Hand sanitizer                 3.00 \
"""}]

panels_v3bis = []
client_prompt_v3bis = pn.widgets.TextInput(value="Hi", placeholder="Enter text here…")


def add_prompts_conversation_v3bis(_):
    prompt = client_prompt_v3bis.value_input
    client_prompt_v3bis.value = ''
    context_v3bis.append({'role': 'user', 'content': f"{prompt}"})
    response = continue_conversation(context_v3bis)
    context_v3bis.append({'role': 'assistant', 'content': f"{response}"})
    panels_v3bis.append(pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels_v3bis.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))
    return pn.Column(*panels_v3bis)


button_v3bis = pn.widgets.Button(name="talk")
interactive_v3bis = pn.bind(add_prompts_conversation_v3bis, button_v3bis)
dashboard_v3bis = pn.Column(
    client_prompt_v3bis,
    pn.Row(button_v3bis),
    pn.panel(interactive_v3bis, loading_indicator=True),
)
dashboard_v3bis

### Comparaison automatique des deux versions

Plutot que de cliquer dans les deux dashboards, on envoie les trois memes questions aux deux
prompts systeme, hors interface. Chaque question part d'un contexte neuf : on teste le prompt
systeme, pas la memoire de conversation.

In [ ]:
questions_test = [
    "Do you have amoxicillin?",
    "What can I take for a migraine?",
    "How much is your melatonin?",
]

def reponse_isolee(system_message, question):
    """Une question, un contexte neuf : on isole l'effet du prompt systeme."""
    return continue_conversation([system_message, {"role": "user", "content": question}])


for q in questions_test:
    print("=" * 90)
    print("QUESTION :", q)
    print("-" * 90)
    print("V3 (prompt d'origine) :\n", reponse_isolee(context_v3[0], q), "\n")
    print("-" * 90)
    print("V3-bis (contraintes ajoutees) :\n", reponse_isolee(context_v3bis[0], q), "\n")

### Recapitulatif de commande en JSON

Le notebook de cours se terminait par un recapitulatif JSON de la commande. C'est aussi un test :
le bot a-t-il correctement suivi ce qui a ete commande, et sait-il additionner ? Le total est
recalcule en Python a partir du JSON pour verifier — c'est le seul moyen de savoir si le modele a
compte juste, plutot que de le croire sur parole.

In [ ]:
import json as _json

def recap_json(contexte):
    """Demande au bot un recapitulatif structure de la commande en cours."""
    messages = contexte.copy()
    messages.append({
        "role": "system",
        "content": (
            "Create a JSON summary of the previous order. Output JSON only, no text around it. "
            "Fields: 1) items: a list of objects with name, size (or null), quantity, unit_price, "
            "line_total 2) total_price."
        ),
    })
    return continue_conversation(messages, temperature=0)


def verifier_total(recap_texte):
    """Recalcule le total en Python : le modele se trompe regulierement sur l'addition."""
    try:
        brut = recap_texte.strip().strip("`")
        brut = brut[brut.index("{"): brut.rindex("}") + 1]
        data = _json.loads(brut)
    except Exception as e:
        return f"JSON illisible ({type(e).__name__}) : le modele n'a pas respecte le format."

    calcule = sum(article.get("line_total", 0) for article in data.get("items", []))
    annonce = data.get("total_price")
    accord = "OK" if annonce is not None and abs(calcule - annonce) < 0.01 else "ECART"
    return f"total annonce = {annonce} | total recalcule = {round(calcule, 2)} -> {accord}"


# A lancer apres avoir passe une commande dans l'un des dashboards ci-dessus.
recap = recap_json(context_v2)      # remplacer par context, context_v3 ou context_v3bis
print(recap)
print()
print(verifier_total(recap))

# Report: Vertical Chat – OrderBot Variations

## Overview
This exercise explored how changing the system prompt transforms
the behavior of the same chatbot architecture into completely
different use cases.

## Version 1 – Fast Food Restaurant (original)
The original OrderBot for "My Dear Frankfurt" worked well.
GPT correctly guided the conversation: greeting, taking the order,
clarifying sizes and toppings, summarizing, and collecting payment.
The friendly tone was consistent throughout.

## Version 2 – Pizza Restaurant (Bella Napoli)
Switching to an Italian pizza restaurant context worked smoothly.
GPT adopted an enthusiastic tone and naturally asked about pizza
size and toppings. No hallucinations were observed — GPT stayed
strictly within the menu provided.

## Version 3 – Pharmacy Assistant (PharmaPlus)
This was the most interesting variation. GPT correctly adopted
a professional and cautious tone, asked about symptoms before
recommending products, and added appropriate disclaimers about
consulting a doctor. This shows how the system prompt can enforce
both tone and ethical boundaries.

## What Didn't Work Well
In Version 3, when asked about prescription medications not on
the menu, GPT occasionally suggested alternatives beyond the
provided list — a mild form of hallucination. This highlights
the importance of explicitly restricting the bot to its menu
in the system prompt.

## Key Learnings
- The **system prompt is the personality** of the bot: changing
  it completely transforms the interaction.
- **Role + constraints + tone** in the system prompt give the
  most predictable results.
- Adding phrases like "only recommend items from the menu"
  reduces hallucinations significantly.
- The conversation memory (the `context` list) is what makes
  multi-turn dialogue possible — GPT has no built-in memory.

---

## Addendum — ce que la V3-bis apporte au rapport

Le rapport ci-dessus conclut qu'ajouter *"only recommend items from the menu"* reduirait les
hallucinations de la V3. La version 3-bis met cette hypothese a l'epreuve au lieu de la poser :
memes questions, memes conditions, un seul element modifie — le prompt systeme.

Trois points a regarder dans la sortie de la cellule de comparaison :

1. **Amoxicilline** (ordonnance, hors catalogue) — la V3 propose souvent une alternative ; la
   V3-bis doit produire la phrase de refus imposee, mot pour mot. Si elle la reformule, c'est
   deja une information : une consigne litterale n'est pas une contrainte, seulement une
   suggestion fortement suivie.
2. **Migraine** — les deux versions devraient rester dans le catalogue (paracetamol, ibuprofene).
   L'ecart interessant est ailleurs : la V3 a tendance a ajouter une posologie, que la V3-bis
   interdit explicitement.
3. **Melatonine** (produit ET prix inexistants) — c'est le test le plus severe. Inventer un prix
   plausible est l'erreur la plus difficile a reperer pour un utilisateur, parce que rien dans la
   reponse ne signale l'invention.

**Ce que ce test ajoute a la conclusion du rapport.** Une regle dans le prompt systeme deplace le
comportement du modele, elle ne le contraint pas : le seul garde-fou reellement fiable est celui
qui ne depend pas du modele — ici, le recalcul du total en Python a partir du JSON. C'est
exactement la distinction entre *demander* un comportement et *verifier* un resultat, et elle
compte d'autant plus que le domaine simule (une pharmacie) est un domaine ou une reponse inventee
a des consequences.

*(Cellules a executer avec une cle OpenAI : noter ici les reponses reellement obtenues pour les
trois questions, dans les deux versions.)*